In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv("../data/processed/harmony_reviews_cleaned.csv")

df.head()

,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,imageURLs,...,reviews.sourceURLs,reviews.text,reviews.title,reviews.username,sourceURLs,upc,weight,original_text,cleaned_text,cleaned_length
0,AVpgfP3DilAPnD_xtG3M,B00IAKLM54,Logitech,"Electronics,Accessories & Supplies,Audio & Vid...",Black,2015-05-18T12:55:52Z,2018-03-23T17:22:18Z,2.2 x 1.2 x 7.3 inches,9.785510e+10,https://images-na.ssl-images-amazon.com/images...,...,https://www.amazon.com/product-reviews/B00IAKL...,"Cannot hit play, FF or Rev, etc, without havin...",Terrible design!,ByC. Anderson,https://www.amazon.com/Logitech-Ultimate-15-De...,9.785510e+10,1.7 pounds,"Cannot hit play, FF or Rev, etc, without havin...",cannot hit play ff or rev etc without having y...,133
1,AVpgfP3DilAPnD_xtG3M,B00IAKLM54,Logitech,"Electronics,Accessories & Supplies,Audio & Vid...",Black,2015-05-18T12:55:52Z,2018-03-23T17:22:18Z,2.2 x 1.2 x 7.3 inches,9.785510e+10,https://images-na.ssl-images-amazon.com/images...,...,https://www.amazon.com/product-reviews/B00IAKL...,HAHAHAHAHA! I'm the master of my own universe ...,End-All-Be-All Fix to remote suffocation,ByAnthony Steiner,https://www.amazon.com/Logitech-Ultimate-15-De...,9.785510e+10,1.7 pounds,HAHAHAHAHA! I'm the master of my own universe ...,hahahahaha i'm the master of my own universe a...,33
2,AVpgfP3DilAPnD_xtG3M,B00IAKLM54,Logitech,"Electronics,Accessories & Supplies,Audio & Vid...",Black,2015-05-18T12:55:52Z,2018-03-23T17:22:18Z,2.2 x 1.2 x 7.3 inches,9.785510e+10,https://images-na.ssl-images-amazon.com/images...,...,https://www.amazon.com/product-reviews/B00IAKL...,I bought this remote to replace my Harmony 650...,Excellent universal remote.,ByMichael R Nielsen,https://www.amazon.com/Logitech-Ultimate-15-De...,9.785510e+10,1.7 pounds,I bought this remote to replace my Harmony 650...,i bought this remote to replace my harmony 650...,56
3,AVpgfP3DilAPnD_xtG3M,B00IAKLM54,Logitech,"Electronics,Accessories & Supplies,Audio & Vid...",Black,2015-05-18T12:55:52Z,2018-03-23T17:22:18Z,2.2 x 1.2 x 7.3 inches,9.785510e+10,https://images-na.ssl-images-amazon.com/images...,...,https://www.amazon.com/product-reviews/B00IAKL...,Universal remote control excellent,Easy to set up and easy to use,BySteve Montealegre,https://www.amazon.com/Logitech-Ultimate-15-De...,9.785510e+10,1.7 pounds,Universal remote control excellent,universal remote control excellent,4
4,AVpgfP3DilAPnD_xtG3M,B00IAKLM54,Logitech,"Electronics,Accessories & Supplies,Audio & Vid...",Black,2015-05-18T12:55:52Z,2018-03-23T17:22:18Z,2.2 x 1.2 x 7.3 inches,9.785510e+10,https://images-na.ssl-images-amazon.com/images...,...,https://www.amazon.com/product-reviews/B00IAKL...,"Other than a few oddly placed buttons, I could...",A ok,ByMarisol,https://www.amazon.com/Logitech-Ultimate-15-De...,9.785510e+10,1.7 pounds,"Other than a few oddly placed buttons, I could...",other than a few oddly placed buttons i couldn...,12


In [3]:
df.shape

(1604, 30)

In [ ]:
#only text column

text_data = df["cleaned_text"]

text_data.head()

0    cannot hit play ff or rev etc without having y...
1    hahahahaha i'm the master of my own universe a...
2    i bought this remote to replace my harmony 650...
3                   universal remote control excellent
4    other than a few oddly placed buttons i couldn...
Name: cleaned_text, dtype: str

In [5]:
#Create the TF-IDF vectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

In [6]:
tfidf_matrix = vectorizer.fit_transform(text_data)

In [7]:
tfidf_matrix.shape

(1604, 5000)

In [8]:
#Look at the vocabulary
feature_names = vectorizer.get_feature_names_out()

feature_names[:20]

array(['00', '000224', '000238', '04', '06', '10', '100', '1000', '1000i',
       '1004', '100x', '104', '105', '1056', '106', '1080p', '10mins',
       '11', '1100', '12'], dtype=object)

In [9]:
len(feature_names)

5000

In [10]:
tfidf_matrix[0].toarray()

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 5000))

In [11]:
feature_names

array(['00', '000224', '000238', ..., 'yr', 'zero', 'zip'],
      shape=(5000,), dtype=object)

In [12]:
#Find the most important words in one review
import numpy as np

row = tfidf_matrix[0].toarray().flatten()

top_indices = row.argsort()[-10:][::-1]

for i in top_indices:
    print(feature_names[i], round(row[i], 3))

arm 0.363
pad 0.238
number 0.205
slightest 0.199
forearm 0.199
brace 0.199
bent 0.199
rev 0.189
flawed 0.189
alphabet 0.189


In [13]:
print(text_data.iloc[0])

cannot hit play ff or rev etc without having your hand arm over the touch pad which is very sensitive the slightest brush can mess up what you are watching i have to brace my forearm on the bottom part keep my wrist bent up very high in order to use the playing options even then it's not a given it won't sense your arm over the top and change it to a different channel or activity also no way to use a number pad that i can find so have to grab another remote to do that it's a nice idea but very flawed design it would be much better to have the touch screen at the top at least and a screen to go to that gives the number alphabet option


In [14]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [15]:
silhouette_scores = {}

for k in range(2, 9):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    cluster_labels = kmeans.fit_predict(tfidf_matrix)
    
    score = silhouette_score(
        tfidf_matrix,
        cluster_labels
    )
    
    silhouette_scores[k] = score
    print(f"K={k}, Silhouette Score={score:.4f}")

K=2, Silhouette Score=0.0061
K=3, Silhouette Score=0.0056
K=4, Silhouette Score=0.0062
K=5, Silhouette Score=0.0053
K=6, Silhouette Score=0.0068
K=7, Silhouette Score=0.0084
K=8, Silhouette Score=0.0089


In [16]:
kmeans = KMeans(
    n_clusters=8,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(tfidf_matrix)

In [17]:
df["cluster"].value_counts().sort_index()

cluster
0    191
1    316
2    102
3    261
4     49
5    420
6    183
7     82
Name: count, dtype: int64

In [18]:
#Find the top words in each cluster
import numpy as np

terms = vectorizer.get_feature_names_out()

for cluster_num in range(8):
    
    cluster_center = kmeans.cluster_centers_[cluster_num]
    
    top_indices = cluster_center.argsort()[-10:][::-1]
    
    top_words = [terms[i] for i in top_indices]
    
    print(f"\nCluster {cluster_num}:")
    print(", ".join(top_words))


Cluster 0:
easy, use, remote, program, set, great, setup, devices, works, product

Cluster 1:
remote, good, harmony, best, logitech, buy, product, works, bought, new

Cluster 2:
love, remote, use, easy, remotes, great, set, screen, having, devices

Cluster 3:
screen, touch, buttons, remote, harmony, like, use, just, logitech, ultimate

Cluster 4:
squad, geek, set, remote, use, easy, tv, did, install, work

Cluster 5:
remotes, devices, harmony, control, works, does, remote, use, work, device

Cluster 6:
tv, remote, cable, turn, box, control, watch, devices, time, player

Cluster 7:
great, works, product, remote, set, recommend, definitely, easy, little, use


In [19]:
#Let's inspect cluster examples
for cluster_num in range(8):
    
    print(f"\n{'='*60}")
    print(f"CLUSTER {cluster_num}")
    print(f"{'='*60}")
    
    samples = df[df["cluster"] == cluster_num]["original_text"].head(3)
    
    for i, review in enumerate(samples, 1):
        print(f"\nReview {i}:")
        print(review)


CLUSTER 0

Review 1:
difficult to use

Review 2:
Easy to setup, controls everything but my Amazon Fire.

Review 3:
Awesome remote, very easy to customize

CLUSTER 1

Review 1:
I bought this remote to replace my Harmony 650 that stop syncing with my devices. It was easy to setup with the Harmony software. I was able to migrate the settings from my Harmony 650 to this remote with no issues. I have been using the remote for almost 2 weeks now and it works perfectly.

Review 2:
Universal remote control excellent

Review 3:
Thought this was similar to my previous Logitech Harmony remote which allowed me to change the channel with the remote pointed in any direction.

CLUSTER 2

Review 1:
Love this. I put all of our other (half dozen) remotes away and we only use this. Even the grandparents can figure it out when they're visiting! Sometimes the macros are slow. Programming takes some time and attention to detail. But the biggest draw back is the rechargeable battery dies every few days! Wou

## Initial Clustering Findings

TF-IDF + K-Means was used as an unsupervised baseline to discover groups
of customer reviews.

The best silhouette score among K=2 to K=8 was obtained at K=8, with a
score of 0.0089. However, the score was very low, indicating weak separation
between clusters.

Inspection of the top cluster terms and sample reviews showed considerable
overlap between clusters. Several clusters were dominated by generic terms
such as "remote", "easy", "great", "use", and "works", rather than clearly
separating specific customer concerns.

This suggests that basic lexical TF-IDF representation is insufficient for
cleanly discovering customer-review topics in this dataset.

The clustering results will therefore be treated as a baseline and further
improved before using LLMs for cluster interpretation.

In [20]:
#Improve the TF-IDF representation
TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",5000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'


In [21]:
vectorizer_v2 = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    min_df=3,
    max_df=0.85
)

In [22]:
tfidf_matrix_v2 = vectorizer_v2.fit_transform(text_data)

print("Matrix shape:", tfidf_matrix_v2.shape)

Matrix shape: (1604, 1999)


In [23]:
feature_names_v2 = vectorizer_v2.get_feature_names_out()

print("Number of features:", len(feature_names_v2))
print(feature_names_v2[:20])

Number of features: 1999
['00' '000224' '10' '100' '1000' '11' '1100' '12' '14' '15' '150' '16'
 '1st' '20' '200' '2015' '2016' '22' '25' '250']


In [24]:
silhouette_scores_v2 = {}

for k in range(2, 9):
    
    kmeans_v2 = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels_v2 = kmeans_v2.fit_predict(tfidf_matrix_v2)
    
    score = silhouette_score(
        tfidf_matrix_v2,
        labels_v2
    )
    
    silhouette_scores_v2[k] = score
    
    print(f"K={k}, Silhouette Score={score:.4f}")

K=2, Silhouette Score=0.0076
K=3, Silhouette Score=0.0079
K=4, Silhouette Score=0.0083
K=5, Silhouette Score=0.0084
K=6, Silhouette Score=0.0073
K=7, Silhouette Score=0.0108
K=8, Silhouette Score=0.0106


In [26]:
vectorizer_v3 = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    min_df=3,
    max_df=0.85,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
)

In [27]:
tfidf_matrix_v3 = vectorizer_v3.fit_transform(text_data)

feature_names_v3 = vectorizer_v3.get_feature_names_out()

print("Matrix shape:", tfidf_matrix_v3.shape)
print("Number of features:", len(feature_names_v3))
print(feature_names_v3[:20])

Matrix shape: (1604, 1951)
Number of features: 1951
['aaa' 'ability' 'able' 'absolute' 'absolutely' 'accept' 'access'
 'accessed' 'accessible' 'accessing' 'accident' 'accidental'
 'accidentally' 'accidently' 'accomplish' 'account' 'accounts' 'accurate'
 'act' 'acting']


In [28]:
silhouette_scores_v3 = {}

for k in range(2, 9):

    kmeans_v3 = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels_v3 = kmeans_v3.fit_predict(tfidf_matrix_v3)

    score = silhouette_score(
        tfidf_matrix_v3,
        labels_v3
    )

    silhouette_scores_v3[k] = score

    print(f"K={k}, Silhouette Score={score:.4f}")

K=2, Silhouette Score=0.0079
K=3, Silhouette Score=0.0081
K=4, Silhouette Score=0.0082
K=5, Silhouette Score=0.0085
K=6, Silhouette Score=0.0088
K=7, Silhouette Score=0.0084
K=8, Silhouette Score=0.0115


In [29]:
kmeans_final = KMeans(
    n_clusters=8,
    random_state=42,
    n_init=10
)

final_labels = kmeans_final.fit_predict(tfidf_matrix_v3)

df["cluster_v3"] = final_labels

In [30]:
terms_v3 = vectorizer_v3.get_feature_names_out()

for cluster_num in range(8):

    cluster_center = kmeans_final.cluster_centers_[cluster_num]

    top_indices = cluster_center.argsort()[-10:][::-1]

    top_words = [terms_v3[i] for i in top_indices]

    print(f"\nCluster {cluster_num}:")
    print(", ".join(top_words))


Cluster 0:
easy, use, remote, set, program, setup, remotes, harmony, devices, control

Cluster 1:
works, remote, great, easy, devices, harmony, program, setup, like, remotes

Cluster 2:
screen, touch, remote, harmony, buttons, like, use, old, just, ultimate

Cluster 3:
squad, geek, set, remote, use, easy, tv, install, work, new

Cluster 4:
remote, good, harmony, remotes, best, control, logitech, work, use, devices

Cluster 5:
love, remote, use, easy, remotes, set, great, screen, having, controls

Cluster 6:
tv, remote, turn, watch, cable, devices, control, box, time, use

Cluster 7:
great, works, product, remote, easy, use, set, recommend, program, definitely


## TF-IDF + K-Means Baseline

Three TF-IDF configurations were evaluated using K-Means clustering.

### Experiment 1 — Basic TF-IDF

- Maximum features: 5,000
- English stopwords removed
- Best K: 8
- Best silhouette score: 0.0089

### Experiment 2 — Frequency filtering

- Maximum features: 5,000
- English stopwords removed
- `min_df=3`
- `max_df=0.85`
- Best K: 7
- Best silhouette score: 0.0108

### Experiment 3 — Frequency filtering + numeric token removal

- Maximum features: 5,000
- English stopwords removed
- `min_df=3`
- `max_df=0.85`
- Numeric-only tokens removed using a custom token pattern
- Best K: 8
- Best silhouette score: 0.0115

### Baseline conclusion

Experiment 3 produced the highest silhouette score, but the absolute
score remained very low.

Qualitative inspection of cluster terms showed substantial overlap
between clusters. Several clusters were dominated by generic product
and sentiment-related terms such as "remote", "easy", "great", "works",
and "use".

Some more specific themes such as touchscreen/buttons and TV/media
device control were visible, but the clustering did not consistently
produce well-separated customer concerns.

Therefore, TF-IDF + K-Means is retained as the traditional NLP baseline,
rather than being treated as the final topic-discovery approach.

In [31]:
from sentence_transformers import SentenceTransformer

In [32]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [33]:
embeddings = embedding_model.encode(
    text_data.tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/51 [00:00<?, ?it/s]

In [34]:
embeddings.shape

(1604, 384)

In [41]:
for k, score in embedding_scores.items():
    print(f"K={k}, Silhouette Score={score:.4f}")

K=2, Silhouette Score=0.1159
K=3, Silhouette Score=0.0262
K=4, Silhouette Score=0.0275
K=5, Silhouette Score=0.0272
K=6, Silhouette Score=0.0354
K=7, Silhouette Score=0.0318
K=8, Silhouette Score=0.0351


In [35]:
#Cluster the embeddings
embedding_scores = {}

for k in range(2, 9):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(embeddings)

    score = silhouette_score(embeddings, labels)

    embedding_scores[k] = score

    print(f"K={k}, Silhouette Score={score:.4f}")

K=2, Silhouette Score=0.1159
K=3, Silhouette Score=0.0262
K=4, Silhouette Score=0.0275
K=5, Silhouette Score=0.0272
K=6, Silhouette Score=0.0354
K=7, Silhouette Score=0.0318
K=8, Silhouette Score=0.0351


In [47]:
best_embedding_kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

df["semantic_cluster_final"] = (
    best_embedding_kmeans.fit_predict(embeddings)
)

In [48]:
print("semantic_cluster_final" in df.columns)

True


In [49]:
df["semantic_cluster"].value_counts().sort_index()

semantic_cluster
0    1119
1     485
Name: count, dtype: int64

In [46]:
#inspect the actual reviews
for cluster_num in [0, 1]:

    print("\n" + "=" * 70)
    print(f"SEMANTIC CLUSTER {cluster_num}")
    print("=" * 70)

    samples = df[
        df["semantic_cluster_final"] == cluster_num
    ]["original_text"].sample(
        min(10, len(df[df["semantic_cluster_final"] == cluster_num])),
        random_state=42
    )

    for i, review in enumerate(samples, 1):
        print(f"\nReview {i}:")
        print(review)


SEMANTIC CLUSTER 0

Review 1:
We bought this remote to replace our Harmony 650. All in all, this is a good remote. If I could, I'd rate 3.5 stars, but I rounded up.The good: Can handle more devices/activities than the 650. The touch screen is clean and responsive, but not overly sensitive so you don't have to worry about accidently launching a command while swiping. Scrolling up/down or side/side is a breeze. It has the most commonly used hard buttons available, so you can perform many functions without needing to look at the remote. Has an internal battery and comes with a charging stand.The bad: It doesn't have a very fluid design at all, nor is it easy to grip if you're trying to use the dvr/dvd controls at the top of the remote. Not sure why they went away from a more traditional remote design, but this was really bad decision making by Logitech.The remote is also kind of slow during the power on phase of various activities. Ex: I use the Watch TV activity to power on my tv/satell

In [51]:
#save
df.to_csv(
    "../data/processed/harmony_reviews_with_clusters.csv",
    index=False
)

print("Saved successfully.")


Saved successfully.


####"The traditional TF-IDF representation produced very weak cluster separation, with a best silhouette score of 0.0115. Sentence embeddings substantially improved the best silhouette score to 0.1159 at K=2, suggesting that semantic representations capture structure in the reviews that lexical TF-IDF does not."